In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, precision_recall_fscore_support, accuracy_score

# ===== CONFIG =====
DATA_DIR = "../data"
LABELS = ["Light", "Medium", "Heavy"]
DPI = 300
FIGSIZE = (10, 8)

# ==================


def compute_confusion_and_metrics(y_true, y_pred):
    """Compute confusion matrix (counts) and metrics."""
    cm = confusion_matrix(y_true, y_pred, labels=[1, 2, 3])
    accuracy = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[1, 2, 3], average="macro", zero_division=0
    )
    return cm, accuracy, precision, recall, f1


def save_confusion_matrix_image(cm, labels, title, save_path, figsize=FIGSIZE):
    """Save confusion matrix image normalized row-wise (percent)."""
    cm_percent = cm.astype(float)
    row_sums = cm_percent.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1  # evita divisione per 0
    cm_percent = (cm_percent / row_sums) * 100

    plt.figure(figsize=figsize)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm_percent, display_labels=labels)
    disp.plot(cmap='Blues', values_format=".2f", colorbar=True, ax=plt.gca())
    disp.im_.set_clim(0, 100)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=DPI)
    plt.close()


def accuracy_score_from_cm(cm):
    """Compute accuracy, precision, recall, f1 from raw confusion matrix (counts)."""
    acc = np.trace(cm) / np.sum(cm) if np.sum(cm) > 0 else 0
    precision, recall, f1 = [], [], []
    for i in range(3):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1_i = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
        precision.append(prec)
        recall.append(rec)
        f1.append(f1_i)
    return acc, np.mean(precision), np.mean(recall), np.mean(f1)


def main():
    subjects = [d for d in os.listdir(DATA_DIR) if d.startswith("S")]
    #subjects = ["S03"]
    print(f"Found subjects: {subjects}")

    for subj in subjects:
        subj_path = os.path.join(DATA_DIR, subj)
        acc_dir = os.path.join(subj_path, "accuracies")
        confusion_dir = os.path.join(acc_dir, "confusion_matrices")
        os.makedirs(acc_dir, exist_ok=True)
        os.makedirs(confusion_dir, exist_ok=True)

        # Aggregation containers
        agg_by_test = {}
        agg_by_box = {"trasp": [], "opaq": []}
        agg_total_myo = []
        agg_total_cv = []

        metrics_rows = []

        # --- find test folders automatically
        tests = [d for d in os.listdir(subj_path) if d.startswith("test")]
        for test in tests:
            agg_by_test[test] = []

            for box in ["trasp", "opaq"]:
                csv_path = os.path.join(subj_path, test, box, "labels.csv")
                if not os.path.exists(csv_path):
                    continue

                df = pd.read_csv(csv_path)
                y_true = df["label_real"].to_numpy()
                y_myo = df["label_myo"].to_numpy()
                y_cv = df["label_cv"].to_numpy()

                # --- MYO ---
                cm_myo, acc_myo, prec_myo, rec_myo, f1_myo = compute_confusion_and_metrics(y_true, y_myo)
                # salva CSV nella stessa cartella dei dati
                np.savetxt(csv_path.replace("labels.csv", "cm_myo.csv"), cm_myo, fmt="%.2f", delimiter=",")
                save_confusion_matrix_image(
                    cm_myo, LABELS, f"{subj} - {test} {box} (MYO) [%]",
                    os.path.join(confusion_dir, f"{test}_{box}_myo.png")
                )
                metrics_rows.append(["MYO", test, box,
                                     round(acc_myo, 2), round(prec_myo, 2), round(rec_myo, 2), round(f1_myo, 2)])
                agg_by_test[test].append((cm_myo, "MYO"))
                agg_by_box[box].append((cm_myo, "MYO"))
                agg_total_myo.append(cm_myo)

                # --- CV ---
                cm_cv, acc_cv, prec_cv, rec_cv, f1_cv = compute_confusion_and_metrics(y_true, y_cv)
                np.savetxt(csv_path.replace("labels.csv", "cm_cv.csv"), cm_cv, fmt="%.2f", delimiter=",")
                save_confusion_matrix_image(
                    cm_cv, LABELS, f"{subj} - {test} {box} (CV) [%]",
                    os.path.join(confusion_dir, f"{test}_{box}_cv.png")
                )
                metrics_rows.append(["CV", test, box,
                                     round(acc_cv, 2), round(prec_cv, 2), round(rec_cv, 2), round(f1_cv, 2)])
                agg_by_test[test].append((cm_cv, "CV"))
                agg_by_box[box].append((cm_cv, "CV"))
                agg_total_cv.append(cm_cv)

        # ==== Aggregated Confusion Matrices ====
        def aggregate_and_save(group_dict, group_name):
            for key, cms in group_dict.items():
                if not cms:
                    continue
                for device in ["MYO", "CV"]:
                    cm_sum = sum(cm for cm, d in cms if d == device)
                    np.savetxt(os.path.join(acc_dir, f"{key}_{device.lower()}_cm.csv"), cm_sum, fmt="%.2f", delimiter=",")
                    save_confusion_matrix_image(cm_sum, LABELS, f"{subj} - {key} ({device}) [%]",
                                                os.path.join(confusion_dir, f"{key}_{device.lower()}.png"))
                    acc, prec, rec, f1 = accuracy_score_from_cm(cm_sum)
                    metrics_rows.append([device, group_name, key,
                                         round(acc, 2), round(prec, 2), round(rec, 2), round(f1, 2)])

        aggregate_and_save(agg_by_test, "test")
        aggregate_and_save(agg_by_box, "box")

        # --- totale soggetto ---
        for device, cm_list in [("MYO", agg_total_myo), ("CV", agg_total_cv)]:
            if cm_list:
                cm_sum = sum(cm_list)
                np.savetxt(os.path.join(acc_dir, f"total_{device.lower()}_cm.csv"), cm_sum, fmt="%.2f", delimiter=",")
                save_confusion_matrix_image(cm_sum, LABELS, f"{subj} - TOTAL ({device}) [%]",
                                            os.path.join(confusion_dir, f"total_{device.lower()}.png"))
                acc, prec, rec, f1 = accuracy_score_from_cm(cm_sum)
                metrics_rows.append([device, "total", "total",
                                     round(acc, 2), round(prec, 2), round(rec, 2), round(f1, 2)])

        # --- salva riepilogo metriche con 2 decimali
        metrics_df = pd.DataFrame(
            metrics_rows,
            columns=["device", "test", "box", "accuracy", "precision", "recall", "f1"]
        )
        metrics_df.to_csv(os.path.join(acc_dir, "summary_metrics.csv"), index=False, float_format="%.2f")
        print(f"✅ Finished {subj}")


if __name__ == "__main__":
    main()


Found subjects: ['S03', 'S08', 'S10', 'S12', 'S13', 'S15', 'S16', 'S17', 'S18', 'S19', 'S20', 'S21']
✅ Finished S03
✅ Finished S08
✅ Finished S10
✅ Finished S12
✅ Finished S13
✅ Finished S15
✅ Finished S16
✅ Finished S17
✅ Finished S18
✅ Finished S19
✅ Finished S20
✅ Finished S21


In [2]:
def main_global():
    
    # --- Setup Global Directories ---
    global_dir = os.path.join(DATA_DIR, "all")
    global_acc_dir = os.path.join(global_dir, "accuracies")
    global_confusion_dir = os.path.join(global_acc_dir, "confusion_matrices")
    global_metrics_by_condition_dir = os.path.join(global_acc_dir, "metrics_by_condition")
    
    os.makedirs(global_acc_dir, exist_ok=True)
    os.makedirs(global_confusion_dir, exist_ok=True)
    os.makedirs(global_metrics_by_condition_dir, exist_ok=True)

    subjects = [d for d in os.listdir(DATA_DIR) if d.startswith("S")]
    print(f"Found subjects: {subjects}")

    all_metrics_dfs = []
    
    # Global Aggregation containers (for raw CMs)
    global_agg_by_test = {}
    global_agg_by_box = {"trasp": [], "opaq": []}
    global_agg_total_myo = []
    global_agg_total_cv = []

    # --- 1. Load all summary_metrics.csv and all raw CMs ---
    for subj in subjects:
        subj_path = os.path.join(DATA_DIR, subj)
        
        # Load summary_metrics.csv
        metrics_csv_path = os.path.join(subj_path, "accuracies", "summary_metrics.csv")
        if os.path.exists(metrics_csv_path):
            try:
                df = pd.read_csv(metrics_csv_path)
                df["subject"] = subj
                all_metrics_dfs.append(df)
            except pd.errors.EmptyDataError:
                print(f"Skipping empty metrics file for {subj}")
        else:
            print(f"Missing summary_metrics.csv for {subj}")

        # Load raw CMs for global aggregation
        tests = [d for d in os.listdir(subj_path) if d.startswith("test")]
        for test in tests:
            if test not in global_agg_by_test:
                global_agg_by_test[test] = [] # Inizializza se non esiste
                
            for box in ["trasp", "opaq"]:
                base_path = os.path.join(subj_path, test, box)
                cm_myo_path = os.path.join(base_path, "cm_myo.csv")
                cm_cv_path = os.path.join(base_path, "cm_cv.csv")

                if os.path.exists(cm_myo_path):
                    try:
                        cm_myo = np.loadtxt(cm_myo_path, delimiter=",")
                        global_agg_by_test[test].append((cm_myo, "MYO"))
                        global_agg_by_box[box].append((cm_myo, "MYO"))
                        global_agg_total_myo.append(cm_myo)
                    except Exception as e:
                        print(f"Error loading {cm_myo_path}: {e}")
                
                if os.path.exists(cm_cv_path):
                    try:
                        cm_cv = np.loadtxt(cm_cv_path, delimiter=",")
                        global_agg_by_test[test].append((cm_cv, "CV"))
                        global_agg_by_box[box].append((cm_cv, "CV"))
                        global_agg_total_cv.append(cm_cv)
                    except Exception as e:
                        print(f"Error loading {cm_cv_path}: {e}")

    if not all_metrics_dfs:
        print("No metrics files found. Exiting.")
        return

    # --- 2. Concatenate all metrics for analysis ---
    all_metrics_df = pd.concat(all_metrics_dfs, ignore_index=True)

    # --- 3. MYO vs CV Wins Analysis ---
    print("Calculating MYO vs CV wins...")
    try:
        pivot_df = all_metrics_df.pivot_table(
            index=["subject", "test", "box"], 
            columns="device", 
            values="accuracy"
        ).reset_index()
        
        pivot_df['winner'] = np.where(
            pivot_df['MYO'] > pivot_df['CV'], 
            'MYO', 
            np.where(pivot_df['CV'] > pivot_df['MYO'], 'CV', 'Tie')
        )
        
        # Conta i vincitori per ogni condizione
        wins_summary = pivot_df.groupby(['test', 'box'])['winner'] \
                               .value_counts() \
                               .unstack(fill_value=0)
                               
        wins_summary_path = os.path.join(global_acc_dir, "myo_vs_cv_wins.csv")
        wins_summary.to_csv(wins_summary_path)
        print(f"Saved MYO vs CV wins summary to {wins_summary_path}")

    except Exception as e:
        print(f"Could not generate MYO vs CV wins summary. Error: {e}")
        print("Pivot head:\n", all_metrics_df.head())


    # --- 4. Save separate CSVs for each condition ---
    print("Saving metrics by condition...")
    grouped = all_metrics_df.groupby(["test", "box"])
    for name, group_df in grouped:
        test_name, box_name = name
        filename = f"{test_name}_{box_name}.csv"
        save_path = os.path.join(global_metrics_by_condition_dir, filename)
        
        # Riorganizza per chiarezza
        try:
            pivoted_group = group_df.pivot_table(
                index="subject", 
                columns=["device"], 
                values=["accuracy", "precision", "recall", "f1"]
            )
            # Riorganizza le colonne per avere device (MYO/CV) al livello superiore
            pivoted_group = pivoted_group.swaplevel(0, 1, axis=1).sort_index(axis=1)
            pivoted_group.to_csv(save_path, float_format="%.2f")
        except:
             # Fallback se il pivot fallisce (es. dati mancanti)
             group_df.to_csv(save_path, index=False, float_format="%.2f")

    print(f"Saved metrics by condition to {global_metrics_by_condition_dir}")


    # --- 5. Calculate and Save Global Aggregated Metrics ---
    print("Calculating global metrics...")
    global_metrics_rows = []

    def aggregate_and_save_global(group_dict, group_name):
        for key, cms in group_dict.items():
            if not cms:
                print(f"No CMs found for group {group_name} - key {key}")
                continue
            for device in ["MYO", "CV"]:
                cm_list_for_device = [cm for cm, d in cms if d == device]
                if not cm_list_for_device:
                    print(f"No CMs found for {device} in group {group_name} - key {key}")
                    continue
                    
                cm_sum = sum(cm_list_for_device)
                
                # Salva CM .csv
                cm_csv_path = os.path.join(global_acc_dir, f"{key}_{device.lower()}_cm.csv")
                np.savetxt(cm_csv_path, cm_sum, fmt="%.2f", delimiter=",")
                
                # Salva Immagine CM
                img_title = f"GLOBAL - {key} ({device}) [%]"
                img_path = os.path.join(global_confusion_dir, f"{key}_{device.lower()}.png")
                save_confusion_matrix_image(cm_sum, LABELS, img_title, img_path)
                
                # Calcola metriche
                acc, prec, rec, f1 = accuracy_score_from_cm(cm_sum)
                global_metrics_rows.append([device, group_name, key,
                                             round(acc, 2), round(prec, 2), round(rec, 2), round(f1, 2)])

    # Aggregati per TEST
    aggregate_and_save_global(global_agg_by_test, "test")
    
    # Aggregati per BOX
    aggregate_and_save_global(global_agg_by_box, "box")

    # Aggregati TOTALI
    for device, cm_list in [("MYO", global_agg_total_myo), ("CV", global_agg_total_cv)]:
        if cm_list:
            cm_sum = sum(cm_list)
            
            # Salva CM .csv
            cm_csv_path = os.path.join(global_acc_dir, f"total_{device.lower()}_cm.csv")
            np.savetxt(cm_csv_path, cm_sum, fmt="%.2f", delimiter=",")
            
            # Salva Immagine CM
            img_title = f"GLOBAL - TOTAL ({device}) [%]"
            img_path = os.path.join(global_confusion_dir, f"total_{device.lower()}.png")
            save_confusion_matrix_image(cm_sum, LABELS, img_title, img_path)
            
            # Calcola metriche
            acc, prec, rec, f1 = accuracy_score_from_cm(cm_sum)
            global_metrics_rows.append([device, "total", "total",
                                         round(acc, 2), round(prec, 2), round(rec, 2), round(f1, 2)])

    # Salva il riepilogo delle metriche GLOBALI
    global_metrics_df = pd.DataFrame(
        global_metrics_rows,
        columns=["device", "test", "box", "accuracy", "precision", "recall", "f1"]
    )
    global_summary_path = os.path.join(global_acc_dir, "summary_metrics.csv")
    global_metrics_df.to_csv(global_summary_path, index=False, float_format="%.2f")
    print(f"✅ Finished global analysis. Results in {global_acc_dir}")


if __name__ == "__main__":
    main_global()

Found subjects: ['S03', 'S08', 'S10', 'S12', 'S13', 'S15', 'S16', 'S17', 'S18', 'S19', 'S20', 'S21']
Calculating MYO vs CV wins...
Saved MYO vs CV wins summary to ../data\all\accuracies\myo_vs_cv_wins.csv
Saving metrics by condition...
Saved metrics by condition to ../data\all\accuracies\metrics_by_condition
Calculating global metrics...
✅ Finished global analysis. Results in ../data\all\accuracies
